In [1]:
import spacy
from Bio import Entrez
import pandas as pd
import re
import ssl
import os
import ollama
import json
ssl._create_default_https_context = ssl._create_unverified_context

In [2]:
# 1. Ask user to input the variant they want to check

# Variant = input("Enter variant (e.g., BRCA1, V600E): ")
variant = "BRCA1 c.5266dupC"

# Option to filter by date
#use_date= input("Filter by year? ")
date_filter = "2020"

# Build search query with date filter
query = f"{variant} AND {date_filter}[PDAT]"

print(f"Gene: {variant}")
print(f"Date filter: {date_filter}")

Gene: BRCA1 c.5266dupC
Date filter: 2020


In [3]:
#2. Download papers from PubMed 

from Bio import Entrez
from Bio import Medline

Entrez.email = "jawahir_noor@hotmail.com"


# Search
handle = Entrez.esearch(db="pubmed", term=query, retmax=10)
record = Entrez.read(handle)
ids = record["IdList"]
handle.close()

# Simpler approach using BioPython's built-in parser
# Fetch in Medline format
handle = Entrez.efetch(db="pubmed", id=ids, rettype="medline", retmode="text")
records = Medline.parse(handle)

papers_list = []
for record in records:
    papers_list.append({
        "title": record.get("TI", ""),
        "abstract": record.get("AB", ""),
        "pmid": record.get("PMID", "")
    })
handle.close()

print(papers_list)



[{'title': 'Prevalence and Spectrum of BRCA Germline Variants in Central Italian High Risk or Familial Breast/Ovarian Cancer Patients: A Monocentric Study.', 'abstract': 'Hereditary breast and ovarian cancers are mainly linked to variants in BRCA1/2 genes. Recently, data has shown that identification of BRCA variants has an immediate impact not only in cancer prevention but also in targeted therapeutic approaches. This prospective observational study characterized the overall germline BRCA variant and variant of uncertain significance (VUS) frequency and spectrum in individuals affected by breast (BC) or ovarian cancer (OC) and in healthy individuals at risk by sequencing the entire BRCA genes. Of the 363 probands analyzed, 50 (13.8%) were BRCA1/2 mutated, 28 (7.7%) at BRCA1 and 23 (6.3%) at BRCA2 gene. The variant c.5266dupC p.(Gln1756Profs) was the most frequent alteration, representing 21.4% of the BRCA1 variants and 12.0% of all variants identified. The variant c.6313delA p.(Ile210

In [4]:
#3. Load model (BioNLP13CG)

# BioNLP model (biomedical entities)
nlp_bionlp = spacy.load("en_ner_bionlp13cg_md")


c:\Users\nicol\OneDrive\Documents\Capstone\.venv\Lib\site-packages\spacy\language.py:2195: FutureWarning: Possible set union at position 6328
  deserializers["tokenizer"] = lambda p: self.tokenizer.from_disk(  # type: ignore[union-attr]


In [5]:
#4. Extract biomedical entities (BioNLP13CG ) paper by paper
import pandas as pd
import re

results = []


for i, paper in enumerate(papers_list):

    text = paper["abstract"]

    if not text:
        continue

    # safe limit
    text = text[:5000]

    
    doc_bio = nlp_bionlp(text)

    # BioNLP entities
    for ent in doc_bio.ents:
        results.append({
            "paper_id": i,
            "title": paper["title"],
            "text": ent.text,
            "label": ent.label_,
            "model": "BioNLP13CG"
        })

The BioNLP13CG model was used to extract biomedical entities such as genes, diseases, and organisms from the collected abstracts. However, this model does not capture structured genomic variant formats (e.g., c.6313delA or p.Arg175His), as it is not specifically trained for mutation recognition. 

To address this limitation, a rule-based approach using regular expressions is implemented to identify variant mentions based on common HGVS-like patterns. The extracted variants are then added to the results as a separate category, allowing the pipeline to combine machine learning-based entity recognition with rule-based mutation extraction for more complete genomic information retrieval.

In [6]:
 # 5. Extract mutation variants (Rule-based + regex) 
mutation_patterns = [
    r"c\.\d+(?:_\d+)?[A-Z]>[A-Z]",        # c.5266C>T  (strict: single base change)
    r"c\.\d+(?:_\d+)?dup[A-Z]?",           # c.5266dupC or c.5266dup
    r"c\.\d+(?:_\d+)?del[A-Z]*",           # c.5266del
    r"c\.\d+(?:_\d+)?ins[A-Z]+",           # c.5266insA
    r"p\.[A-Z][a-z]{2}\d+[A-Z][a-z]{2}",  # p.Gln1756Ter  (strict 3-letter AA)
    r"rs\d{4,}",                            # rs ID (min 4 digits to avoid false hits)
]

def is_valid_variant(v):
    """Reject malformed extractions like 'c1016d', single-letter hits, etc."""
    # Must start with c., p., or rs
    if not re.match(r"^(c\.|p\.|rs)", v, re.IGNORECASE):
        return False
    # c. variants must have a numeric position
    if v.startswith("c.") and not re.search(r"c\.\d+", v):
        return False
    return True
 
def extract_variants(text):
    variants = []
    for pattern in mutation_patterns:
        matches = re.findall(pattern, text)
        variants.extend(matches)
    return [v for v in set(variants) if is_valid_variant(v)]
 
variant_results = []
for i, paper in enumerate(papers_list):
    text = paper["abstract"]
    if not text:
        continue
    for variant_match in extract_variants(text[:5000]):
        variant_results.append({
            "paper_id": i,
            "title": paper["title"],
            "pmid": paper.get("pmid", ""),
            "variant": variant_match
        })

In [7]:

# 6. LitVar2 normalization

LITVAR2_BASE = "https://www.ncbi.nlm.nih.gov/research/litvar2-api"
 
def litvar2_normalize(variant_name, gene_name):
    """
    Normalize a variant using the LitVar2 API.
 
    Step 1 — autocomplete: query LitVar2 with the variant string.
              Returns a list of candidates, each with rsid, gene(s), pmids_count.
    Step 2 — pick the best candidate that matches our gene.
    Step 3 — fetch linked PMIDs for that rsid.
 
    Returns a dict with: rsid, litvar_name, gene, pmids_count, pmids (list)
    """
    try:
        # Step 1: autocomplete search
        url = f"{LITVAR2_BASE}/variant/autocomplete/?query={quote(variant_name)}"
        r = requests.get(url, timeout=15)
        time.sleep(0.34)
 
        if r.status_code != 200 or not r.json():
            return None
 
        candidates = r.json()
 
        # Step 2: pick best candidate matching our gene
        best = None
        for candidate in candidates:
            candidate_genes = candidate.get("gene", [])
            # Check if our gene appears in the candidate's gene list
            if any(gene_name.upper() == g.upper() for g in candidate_genes):
                best = candidate
                break
 
        # If no gene match, take the first result (may still be relevant)
        if best is None and candidates:
            best = candidates[0]
 
        if best is None:
            return None
 
        rsid        = best.get("rsid", "N/A")
        litvar_name = best.get("name", variant_name)
        genes       = best.get("gene", [])
        pmids_count = best.get("pmids_count", 0)
 
        # Step 3: fetch linked PMIDs for this rsid
        pmids = []
        if rsid != "N/A" and int(pmids_count) > 0:
            encoded_id = f"litvar%40{rsid}%23%23"
            pub_url = f"{LITVAR2_BASE}/variant/get/{encoded_id}/publications"
            r2 = requests.get(pub_url, timeout=15)
            time.sleep(0.34)
            if r2.status_code == 200:
                pub_data = r2.json()
                pmids = pub_data.get("pmids", [])
 
        return {
            "rsid":        rsid,
            "litvar_name": litvar_name,
            "gene":        ", ".join(genes) if genes else gene_name,
            "pmids_count": pmids_count,
            "pmids":       pmids,
        }
 
    except Exception:
        return None


In [8]:
# 6.b. Build tables, normalize, print and save

os.makedirs("results", exist_ok=True)
 
df_entities = pd.DataFrame(results)
# Define gene from your variant
gene = variant.split()[0]
 
# Normalize via LitVar2 and add columns directly into variant_results rows
print("\nNormalizing variants via LitVar2...")
for row in variant_results:
    result = litvar2_normalize(row["variant"], gene)
 
    if result:
        row["rsid"]        = result["rsid"]
        row["litvar_name"] = result["litvar_name"]
        row["pmids_count"] = result["pmids_count"]
        row["litvar_pmids"] = "|".join(str(p) for p in result["pmids"][:5])  # first 5 PMIDs
        print(f"  {row['variant']} → {result['rsid']} | {result['litvar_name']} | {result['pmids_count']} papers")
    else:
        row["rsid"]        = "N/A"
        row["litvar_name"] = "N/A"
        row["pmids_count"] = 0
        row["litvar_pmids"] = "N/A"
        print(f"  {row['variant']} → Not found in LitVar2")
 
df_variants = pd.DataFrame(variant_results)
 
# Print 
print("\n=== Entities Found ===")
print(df_entities.head())
print(f"\nTotal entities: {len(df_entities)}")
 
print("\n=== Variants Found ===")
print(df_variants[["pmid", "variant", "rsid", "litvar_name", "pmids_count", "litvar_pmids"]].head(20))
print(f"\nTotal variants: {len(df_variants)}")
 
# Save 
df_entities.to_csv("results/paper_by_paper_entities.csv", index=False)
df_variants.to_csv("results/extracted_variants.csv", index=False)
 
print("\nResults saved to 'results/' directory")



Normalizing variants via LitVar2...
  c.6313delA → Not found in LitVar2
  c.5266dupC → Not found in LitVar2
  c.5266dupC → Not found in LitVar2
  c.5946delT → Not found in LitVar2
  c.5266dupC → Not found in LitVar2
  c.68_69delAG → Not found in LitVar2
  c.5266dupC → Not found in LitVar2
  c.5266dupC → Not found in LitVar2
  c.8940delA → Not found in LitVar2
  c.9097dupA → Not found in LitVar2
  c.5946delT → Not found in LitVar2
  c.5266dupC → Not found in LitVar2
  c.68_69delAG → Not found in LitVar2

=== Entities Found ===
   paper_id                                              title  \
0         0  Prevalence and Spectrum of BRCA Germline Varia...   
1         0  Prevalence and Spectrum of BRCA Germline Varia...   
2         0  Prevalence and Spectrum of BRCA Germline Varia...   
3         0  Prevalence and Spectrum of BRCA Germline Varia...   
4         0  Prevalence and Spectrum of BRCA Germline Varia...   

              text                 label       model  
0           bre

In [9]:
# 7. Evaluation matrix

from sklearn.metrics import precision_score, recall_score, f1_score

# What the pipeline extracted
found = {
    0: ["c.5266dupC", "c.3607C>T", "c.9371A>T"],  # Paper 0 - Romanian study
    1: ["c.5266dupC", "c.181T>G", "c.211dupA","c.181T>G","c.68_69del","c.798_799delTT"],   # Paper 1 - North African study
}

# Gold standard (what found in each paper)
correct = {
    0: ["c.5266dupC", "c.3607C>T", "c.9371A>T", "c.181T>G","c.1687C>T", "c.68_69delAG", "c.4218delG"],
    1: ["c.211dupA", "c.798_799delTT", "c.5266dupC", "c.5309G>T","c.3279delC", "c.1310_1313delAAGA", "c.68_69delAG", "c.181T>G"]
}

y_true = [1, 1]  # papers actually have it 1=has mutation, 0=no mutation
y_pred = [1, 1]  # the model detected

precision = precision_score(y_true, y_pred)
recall = recall_score(y_true, y_pred)
f1 = f1_score(y_true, y_pred)

print(f"Precision: {precision:.1%}")
print(f"Recall: {recall:.1%}")
print(f"F1: {f1:.1%}")

Precision: 100.0%
Recall: 100.0%
F1: 100.0%


In [ ]:
#8 - LLM Extraction of Clinical Details (Classification, Zygosity, Study Type, Patient Count)

def get_clinical_details(abstract_text):
    prompt = f"""
    Analyze the abstract and extract:
    - Classification (Pathogenic, Benign, VUS)
    - Zygosity (Heterozygous, Homozygous)
    - Study type (Case-control, Cohort, functional etc.)
    - Patient count
    - Functional evidence (in vitro, splicing, etc.)
    - Publication year
    
    Abstract: {abstract_text}

    Return only a valid JSON object with keys: 
    "classification", "zygosity", "study_type", "patient_count", "functional_evidence", "publication_year"
    """
    response = ollama.chat(
        model='llama3', 
        messages=[{"role": "user", "content": prompt}],
        options={"num_ctx": 2048, "temperature": 0} 
    )
    
    content = response['message']['content']
    
    try:
        json_match = re.search(r'\{.*\}', content, re.DOTALL)
        if json_match:
            return json.loads(json_match.group())
        return {} 
    except:
        return {}

output = []

for paper in papers_list:
    abstract = paper.get("abstract")
    if not abstract: continue

    print(f"Processing: {paper.get('pmid', 'Unknown')}...")
    
    # Get the LLM details
    details = get_clinical_details(abstract)
    
    combined_entry = {
        "pmid": paper.get("pmid"),
        "gene": paper.get("gene"),
        "hgvs_cdna": paper.get("hgvs_cdna"),
        "classification": details.get("classification", "N/A"),
        "zygosity": details.get("zygosity", "N/A"),
        "study_type": details.get("study_type", "N/A"),
        "patient_count": details.get("patient_count", "N/A")
    }
    
    output.append(combined_entry)

file_name = "clinical_extraction_results.json"
with open(file_name, 'w', encoding='utf-8') as f:
    json.dump(output, f, indent=4)

df_clinical = pd.DataFrame(output)
print("\nFinal Output Preview:")
print(df_clinical.head())


Processing: 32806537...
